# 53 seisan rea diagnostics

53_seisan_rea_diagnostics.py

STEP 53 — Diagnostics and sanity checks for SEISAN REA catalog.

Uses EVENT_CLASS comments written in Step 52 as the authoritative source
of event composition (W, P, H), with heuristic fallback for legacy files.

Outputs:
  • Summary counts by EVENT_CLASS
  • Daily counts:
      - total events
      - events with waveforms
      - events with picks
      - events with hypocenters
  • Daily EVENT_CLASS counts
  • CSV + PNG plot

Author: GT__/ChatGPT

## Scope and execution

Uses the script’s actual REA/out-dir interface; the obsolete --db-name launcher option is omitted.

Review `DATA_TOP` and `SETTINGS`, then run the processing cell. Outputs may overwrite existing products according to the original step. The implementation is in the adjacent local module; no command-line script or subprocess is invoked.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / 'pipeline', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths
from pinatubo_steps.step_53_seisan_rea_diagnostics import run

DATA_TOP = Path.home() / 'Dropbox/PROFESSIONAL/DATA/Pinatubo/ASSEMBLED-25-028-THOMPSON-PINATUBO1991'
P = pipeline_paths(DATA_TOP)

SETTINGS = dict(
    rea_dir=P['FAIR_REA_DIR'],
    out_dir=P['SEISAN_DIAG_DIR'],
    wavefile_regex='\\b[MS]\\.[A-Za-z0-9]{1,12}_[A-Za-z0-9]{1,6}\\b',
)
display(SETTINGS)


## Run the step

Imports and settings above do not run the analysis. This cell writes the configured products.


In [ ]:
result = run(**SETTINGS)


In [ ]:
# Preview returned tables and summary objects without reloading output files.
for name, value in result.items():
    print(name)
    display(value.head() if hasattr(value, 'head') else
            value[:5] if isinstance(value, list) else value)


In [ ]:
from IPython.display import Image

# Show saved diagnostics inline; all figures remain in the configured folder.
plot_roots = [Path(SETTINGS[k]) for k in ('outdir', 'out_dir', 'out')
              if k in SETTINGS and SETTINGS[k] is not None]
plots = sorted({p for root in plot_roots
                for p in ([root] if root.suffix.lower() == '.png'
                          else root.rglob('*.png')) if p.is_file()})
print(f'{len(plots)} saved PNG figures; showing up to four.')
for plot in plots[:4]:
    print(plot)
    display(Image(filename=str(plot)))
